# Noise Robustness for ML-QRC

This notebook implements the **Noise Robustness** subsection from `ML_QRC-16.pdf`.

The experiment uses the Mackey-Glass forecasting task and sweeps the depolarizing-noise level

\[
p_{\mathrm{dep}} \in \{0, 0.005, 0.01, 0.02, 0.05\}.
\]

At each noise level, the notebook compares:

- **Ridge** on flattened QRC features
- **ML-QRC-M** with multilinear width `d = 8`

The notebook follows the Task 1 protocol:

- `n = 6` qubits
- lookback `T = 50`
- Mackey-Glass `1000 / 3000 / 1000` washout/train/test split
- full-data regime only
- `5` evaluation seeds

All readouts at a given `p_dep` use the **same cached QRC tensor bank**, so the only difference is the readout.


In [ ]:
# This cell installs the small package set used by the notebook.

%pip install -q numpy pandas matplotlib torch pennylane jupyter


In [ ]:
# This cell imports the libraries, resolves the repository root, and
# prepares the artifact directory used by this notebook.

from pathlib import Path
import os
import random
import time

here = Path.cwd()
if (here / "scripts").exists():
    ROOT = here
elif (here.parent / "scripts").exists():
    ROOT = here.parent
else:
    ROOT = here

cache_root = ROOT / "notebooks" / "artifacts" / "cache"
cache_root.mkdir(parents=True, exist_ok=True)
(cache_root / "mplconfig").mkdir(parents=True, exist_ok=True)

os.environ["XDG_CACHE_HOME"] = str(cache_root.resolve())
os.environ["MPLCONFIGDIR"] = str((cache_root / "mplconfig").resolve())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams["figure.dpi"] = 1000
plt.rcParams["savefig.dpi"] = 1000
plt.rcParams["figure.figsize"] = (3, 2.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.precision", 6)
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {DEVICE}")

ARTIFACT_DIR = ROOT / "notebooks" / "artifacts" / "noise_robustness"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Repo root: {ROOT.resolve()}")
print(f"Artifacts: {ARTIFACT_DIR.resolve()}")
print(f"PyTorch: {torch.__version__}")
print(f"PennyLane: {qml.__version__}")


In [ ]:
# This cell stores the experiment configuration in one place.

CONFIG = {
    "eval_seeds": list(range(5)),
    "reservoir_seed": 42,
    "noise_levels": [0.0, 0.005, 0.01, 0.02, 0.05],
    "num_points": 5000,
    "washout": 1000,
    "train_pool": 3000,
    "test_size": 1000,
    "mg_beta": 0.2,
    "mg_gamma": 0.1,
    "mg_q": 10,
    "mg_tau": 17.0,
    "mg_step": 0.1,
    "mg_x0": 1.2,
    "n_qubits": 6,
    "n_layers": 1,
    "lookback": 50,
    "qrc_batch": 256,
    "epochs": 300,
    "batch_size": 32,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "mlqrc_d": 8,
    "ridge_alphas": (1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5),
}

N_Q = CONFIG["n_qubits"]
T = CONFIG["lookback"]
F_LOCAL = N_Q
F_PAIR = N_Q * (N_Q - 1) // 2
F = F_LOCAL + F_PAIR

display(pd.Series(CONFIG, name="value"))
print(f"Observable channels: F = {F}  (Z = {F_LOCAL}, ZZ = {F_PAIR})")


In [ ]:
# This cell recreates the Mackey-Glass sequence and the same supervised
# one-step-ahead split used by the main Task 1 notebook.

def generate_mackey_glass(num_points, beta, gamma, q, tau, step, x0):
    delay = int(round(tau / step))
    total = delay + num_points
    x = np.full(total + 1, x0, dtype=np.float64)

    def rhs(x_now, x_delay):
        return beta * x_delay / (1.0 + x_delay ** q) - gamma * x_now

    for k in range(delay, total):
        x_now = x[k]
        x_d0 = x[k - delay]
        x_dh = 0.5 * (x[k - delay] + x[k - delay + 1])
        x_d1 = x[k - delay + 1]
        k1 = rhs(x_now, x_d0)
        k2 = rhs(x_now + 0.5 * step * k1, x_dh)
        k3 = rhs(x_now + 0.5 * step * k2, x_dh)
        k4 = rhs(x_now + step * k3, x_d1)
        x[k + 1] = x_now + (step / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

    return x[delay + 1 : delay + 1 + num_points].astype(np.float32)

mg = generate_mackey_glass(
    CONFIG["num_points"],
    CONFIG["mg_beta"],
    CONFIG["mg_gamma"],
    CONFIG["mg_q"],
    CONFIG["mg_tau"],
    CONFIG["mg_step"],
    CONFIG["mg_x0"],
)

W = CONFIG["washout"]
TR = CONFIG["train_pool"]
TE = CONFIG["test_size"]

train_ref = mg[W : W + TR]
scale_min = float(train_ref.min())
scale_range = float(train_ref.max() - train_ref.min())
mg_sc = ((mg - scale_min) / scale_range).astype(np.float32)

def unscale(values):
    return (np.asarray(values, dtype=np.float64) * scale_range + scale_min).astype(np.float32)

target_times = np.arange(T, len(mg), dtype=np.int32)
input_windows = np.stack(
    [mg_sc[idx - T : idx] for idx in target_times]
).astype(np.float32)
y_sc = mg_sc[target_times].astype(np.float32)
y_orig = mg[target_times].astype(np.float32)

tr_pos = np.where((target_times >= W) & (target_times < W + TR))[0]
te_pos = np.where((target_times >= W + TR) & (target_times < W + TR + TE))[0]

assert len(tr_pos) == TR
assert len(te_pos) == TE

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=1000)
ax.plot(mg, color="black", linewidth=0.6)
ax.axvspan(0, W, color="#d9d9d9", alpha=0.45, label="Washout")
ax.axvspan(W, W + TR, color="#c7e9c0", alpha=0.35, label="Train")
ax.axvspan(W + TR, W + TR + TE, color="#fdd0a2", alpha=0.35, label="Test")
ax.set_title("Mackey-Glass Split")
ax.set_xlabel("Time index")
ax.set_ylabel("Value")
ax.legend(fontsize=5, frameon=False, ncol=2)
plt.tight_layout()
plt.show()

print(f"Input windows: {input_windows.shape}")
print(f"Training samples: {len(tr_pos)}")
print(f"Test samples: {len(te_pos)}")


## Noise Model Used Here

For this notebook, the noise sweep uses a **stochastic single-qubit Pauli-error rollout**.

At each reservoir step and for each qubit independently:

- with probability `1 - p_dep`, nothing happens,
- with probability `p_dep / 3`, apply `X`,
- with probability `p_dep / 3`, apply `Y`,
- with probability `p_dep / 3`, apply `Z`.

This is a Monte Carlo pure-state surrogate for depolarizing noise. It is more informative for a robustness sweep than
simple post-measurement rescaling, because it perturbs the reservoir trajectory itself and therefore cannot be removed by
per-feature standardization.

Each noise level is cached separately with a fixed noise RNG seed, so reruns are deterministic.


In [ ]:
# This cell builds and caches the noisy QRC tensor banks.
# The stochastic Pauli errors are fixed by a deterministic RNG seed per p_dep.

set_seed(CONFIG["reservoir_seed"])
rng_res = np.random.default_rng(CONFIG["reservoir_seed"])
res_W = rng_res.uniform(
    0.0,
    2.0 * np.pi,
    size=(CONFIG["n_layers"], N_Q, 3),
).astype(np.float32)

res_U = torch.tensor(
    qml.matrix(
        qml.StronglyEntanglingLayers(res_W, wires=range(N_Q)),
        wire_order=range(N_Q),
    ),
    dtype=torch.complex64,
    device=DEVICE,
)

def obs_mat(op):
    return torch.tensor(qml.matrix(op, wire_order=range(N_Q)), dtype=torch.complex64, device=DEVICE)

obs_mats = [obs_mat(qml.PauliZ(i)) for i in range(N_Q)]
for i in range(N_Q):
    for j in range(i + 1, N_Q):
        obs_mats.append(obs_mat(qml.PauliZ(i) @ qml.PauliZ(j)))

pauli_gates = torch.tensor(
    [
        [[1.0 + 0.0j, 0.0 + 0.0j], [0.0 + 0.0j, 1.0 + 0.0j]],
        [[0.0 + 0.0j, 1.0 + 0.0j], [1.0 + 0.0j, 0.0 + 0.0j]],
        [[0.0 + 0.0j, -1.0j], [1.0j, 0.0 + 0.0j]],
        [[1.0 + 0.0j, 0.0 + 0.0j], [0.0 + 0.0j, -1.0 + 0.0j]],
    ],
    dtype=torch.complex64,
    device=DEVICE,
)

zero_st = torch.zeros(2 ** N_Q, dtype=torch.complex64, device=DEVICE)
zero_st[0] = 1.0 + 0.0j
pi_f = torch.tensor(float(np.pi), dtype=torch.float32, device=DEVICE)

def batched_ry(states, angles, wire):
    c = torch.cos(angles / 2.0).to(torch.complex64)
    s = torch.sin(angles / 2.0).to(torch.complex64)
    gate = torch.stack(
        [
            torch.stack([c, -s], dim=-1),
            torch.stack([s, c], dim=-1),
        ],
        dim=-2,
    )
    state_view = states.reshape(states.shape[0], *([2] * N_Q))
    state_view = torch.movedim(state_view, 1 + wire, -1)
    state_view = torch.einsum("bij,b...j->b...i", gate, state_view)
    state_view = torch.movedim(state_view, -1, 1 + wire)
    return state_view.reshape(states.shape[0], -1)

def apply_batched_gate(states, gate_batch, wire):
    state_view = states.reshape(states.shape[0], *([2] * N_Q))
    state_view = torch.movedim(state_view, 1 + wire, -1)
    state_view = torch.einsum("bij,b...j->b...i", gate_batch, state_view)
    state_view = torch.movedim(state_view, -1, 1 + wire)
    return state_view.reshape(states.shape[0], -1)

def build_qrc_features(windows, p_dep):
    cache = ARTIFACT_DIR / (
        f"qrc_feats_mcpauli_v1_rseed{CONFIG['reservoir_seed']}_p{p_dep:.3f}_q{N_Q}_T{T}.npy"
    )
    if cache.exists():
        print(f"Loading noisy QRC features: {cache.name}")
        return np.load(cache)

    wt = torch.tensor(windows, dtype=torch.float32, device=DEVICE)
    out_chunks = []
    t0 = time.time()
    noise_seed = CONFIG["reservoir_seed"] * 1000 + int(round(p_dep * 1000))
    rng_noise = np.random.default_rng(noise_seed)

    for start in range(0, len(wt), CONFIG["qrc_batch"]):
        batch = wt[start : start + CONFIG["qrc_batch"]]
        bs = batch.shape[0]

        states = zero_st.unsqueeze(0).expand(bs, -1).clone()
        feats = torch.zeros(bs, T, N_Q, F, dtype=torch.float32, device=DEVICE)
        qi = torch.arange(N_Q, device=DEVICE)

        for t_step in range(T):
            u = batch[:, t_step]
            angles = pi_f * u
            for wire in range(N_Q):
                states = batched_ry(states, angles, wire)

            states = states @ res_U.T

            if p_dep > 0.0:
                probs = np.array(
                    [1.0 - p_dep, p_dep / 3.0, p_dep / 3.0, p_dep / 3.0],
                    dtype=np.float64,
                )
                for wire in range(N_Q):
                    codes = rng_noise.choice(4, size=bs, p=probs)
                    gate_batch = pauli_gates[torch.tensor(codes, dtype=torch.long, device=DEVICE)]
                    states = apply_batched_gate(states, gate_batch, wire)

            evs = torch.stack(
                [
                    torch.real(torch.sum(states.conj() * (states @ op.T), dim=1))
                    for op in obs_mats
                ],
                dim=1,
            )

            feats[:, t_step, qi, qi] = evs[:, :F_LOCAL]
            pair_idx = 0
            for i in range(N_Q):
                for j in range(i + 1, N_Q):
                    col = F_LOCAL + pair_idx
                    feats[:, t_step, i, col] = evs[:, col]
                    feats[:, t_step, j, col] = evs[:, col]
                    pair_idx += 1

        out_chunks.append(feats.cpu().numpy())
        done = min(start + CONFIG["qrc_batch"], len(wt))
        if done % 1024 == 0 or done == len(wt):
            print(f"  built {done}/{len(wt)} windows  ({time.time() - t0:.0f}s)")

    out = np.concatenate(out_chunks, axis=0).astype(np.float32)
    np.save(cache, out)
    print(f"Saved noisy QRC features to: {cache.resolve()}")
    return out

qrc_by_noise = {}
for p_dep in CONFIG["noise_levels"]:
    qrc_by_noise[p_dep] = build_qrc_features(input_windows, p_dep)

flat_dim = int(np.prod(qrc_by_noise[CONFIG["noise_levels"][0]].shape[1:]))
print(f"Reference QRC tensor bank: {qrc_by_noise[CONFIG['noise_levels'][0]].shape}")
print(f"Flattened feature dimension: {flat_dim}")


In [ ]:
# This cell confirms that every requested noise level has a cached QRC tensor bank.

print("Cached noise levels:", ", ".join(f"{p:.3f}" for p in CONFIG["noise_levels"]))
for p_dep in CONFIG["noise_levels"]:
    print(f"  p_dep={p_dep:.3f}  shape={qrc_by_noise[p_dep].shape}")


In [ ]:
# This cell defines the metric, preprocessing, and model helpers used
# by the noise sweep.

def nmse(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.float64)
    yp = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum((yt - yp) ** 2) / np.sum((yt - yt.mean()) ** 2))

class Standardizer:
    def fit(self, features):
        flat = features.reshape(features.shape[0], -1).astype(np.float64)
        self.mu = flat.mean(axis=0, keepdims=True)
        self.sig = flat.std(axis=0, keepdims=True)
        self.sig[self.sig < 1e-8] = 1.0
        return self

    def transform(self, features):
        shape = features.shape
        flat = features.reshape(shape[0], -1).astype(np.float64)
        flat = (flat - self.mu) / self.sig
        return flat.reshape(shape).astype(np.float32)

def fit_ols(Xtr, ytr):
    design = np.hstack([Xtr, np.ones((len(Xtr), 1), dtype=np.float32)])
    return np.linalg.lstsq(design, ytr, rcond=None)[0].astype(np.float32)

def predict_ols(weights, X):
    design = np.hstack([X, np.ones((len(X), 1), dtype=np.float32)])
    return (design @ weights).astype(np.float32)

def fit_ridge_cv(Xtr, ytr, alphas, val_frac=0.2):
    split = max(1, int(round(len(Xtr) * (1.0 - val_frac))))
    Xt = Xtr[:split].astype(np.float64)
    yt = ytr[:split].astype(np.float64)
    Xv = Xtr[split:].astype(np.float64)
    yv = ytr[split:].astype(np.float64)

    mux = Xt.mean(axis=0)
    muy = float(yt.mean())
    Xc = Xt - mux
    yc = yt - muy

    best_alpha = alphas[0]
    best_score = np.inf

    for alpha in alphas:
        lam = np.linalg.solve(Xc @ Xc.T + alpha * np.eye(len(Xt)), yc)
        weights = Xc.T @ lam
        bias = muy - mux @ weights
        score = nmse(yv, Xv @ weights + bias)
        if score < best_score:
            best_score = score
            best_alpha = alpha

    mux = Xtr.astype(np.float64).mean(axis=0)
    muy = float(ytr.mean())
    Xc = Xtr.astype(np.float64) - mux
    yc = ytr.astype(np.float64) - muy
    lam = np.linalg.solve(Xc @ Xc.T + best_alpha * np.eye(len(Xtr)), yc)
    weights = Xc.T @ lam
    bias = muy - mux @ weights
    return np.append(weights, bias).astype(np.float32), float(best_alpha)

class MLQRCReadout(nn.Module):
    def __init__(self, time_steps, qubits, observables, d):
        super().__init__()
        self.U_T = nn.Parameter(torch.randn(d, time_steps) * 0.05)
        self.U_n = nn.Parameter(torch.randn(d, qubits) * 0.05)
        self.U_F = nn.Parameter(torch.randn(d, observables) * 0.05)
        self.head = nn.Linear(d * d * d, 1)

    def forward(self, x):
        y = torch.einsum("btnf,dt->bdnf", x, self.U_T)
        y = torch.einsum("bdnf,in->bdif", y, self.U_n)
        y = torch.einsum("bdif,jf->bdij", y, self.U_F)
        return self.head(y.reshape(y.shape[0], -1)).squeeze(-1)

def count_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def train_model(model, Xtr, ytr, epochs, batch_size, lr, wd):
    model = model.to(DEVICE)
    dataset = TensorDataset(
        torch.tensor(Xtr, dtype=torch.float32),
        torch.tensor(ytr, dtype=torch.float32),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    model.train()

    for _ in range(epochs):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

    return model

@torch.no_grad()
def predict_model(model, X):
    model.eval()
    return model(torch.tensor(X, dtype=torch.float32, device=DEVICE)).cpu().numpy().astype(np.float32)

print(f"ML-QRC-M parameter count: {count_params(MLQRCReadout(T, N_Q, F, CONFIG['mlqrc_d']))}")


In [ ]:
# This cell runs the full noise sweep.
# Ridge is deterministic for a fixed feature bank, so its value is the same
# for every seed. The notebook still records one row per seed so the summary
# table has the same schema for both methods.

result_tag = "mcpauli_v1"
raw_cache_csv = ARTIFACT_DIR / f"noise_robustness_raw_{result_tag}.csv"
summary_cache_csv = ARTIFACT_DIR / f"noise_robustness_summary_{result_tag}.csv"
wide_cache_csv = ARTIFACT_DIR / f"table_noise_{result_tag}.csv"

raw_csv = ARTIFACT_DIR / "noise_robustness_raw.csv"
summary_csv = ARTIFACT_DIR / "noise_robustness_summary.csv"
wide_csv = ARTIFACT_DIR / "table_noise.csv"

if raw_cache_csv.exists() and summary_cache_csv.exists() and wide_cache_csv.exists():
    raw_results = pd.read_csv(raw_cache_csv)
    noise_summary = pd.read_csv(summary_cache_csv)
    noise_table = pd.read_csv(wide_cache_csv)
    print("Loaded cached noise-robustness results.")
else:
    rows = []
    train_index = tr_pos[: CONFIG["train_pool"]]
    test_index = te_pos
    ytr = y_sc[train_index]
    yte = y_orig[test_index]

    t0 = time.time()
    total_jobs = len(CONFIG["noise_levels"]) * (1 + len(CONFIG["eval_seeds"]))
    job_done = 0

    for p_dep in CONFIG["noise_levels"]:
        feats = qrc_by_noise[p_dep]
        scaler = Standardizer().fit(feats[train_index])
        Xtr_t = scaler.transform(feats[train_index])
        Xte_t = scaler.transform(feats[test_index])
        Xtr_f = Xtr_t.reshape(len(train_index), -1)
        Xte_f = Xte_t.reshape(len(test_index), -1)

        ridge_w, ridge_alpha = fit_ridge_cv(Xtr_f, ytr, CONFIG["ridge_alphas"])
        ridge_pred = unscale(predict_ols(ridge_w, Xte_f))
        ridge_nmse = nmse(yte, ridge_pred)
        for seed in CONFIG["eval_seeds"]:
            rows.append(
                {"p_dep": p_dep, "seed": seed, "model": "Ridge", "nmse": ridge_nmse}
            )
        job_done += 1
        print(
            f"[{job_done:02d}/{total_jobs:02d}] p_dep={p_dep:.3f}  "
            f"Ridge NMSE={ridge_nmse:.6f}  alpha={ridge_alpha:g}"
        )

        for seed in CONFIG["eval_seeds"]:
            set_seed(seed)
            model = MLQRCReadout(T, N_Q, F, CONFIG["mlqrc_d"])
            model = train_model(
                model,
                Xtr_t,
                ytr,
                CONFIG["epochs"],
                CONFIG["batch_size"],
                CONFIG["lr"],
                CONFIG["weight_decay"],
            )
            pred = unscale(predict_model(model, Xte_t))
            score = nmse(yte, pred)
            rows.append(
                {"p_dep": p_dep, "seed": seed, "model": "ML-QRC-M", "nmse": score}
            )
            job_done += 1
            print(
                f"[{job_done:02d}/{total_jobs:02d}] p_dep={p_dep:.3f}  "
                f"seed={seed}  ML-QRC-M NMSE={score:.6f}"
            )

    raw_results = pd.DataFrame(rows)
    noise_summary = (
        raw_results
        .groupby(["p_dep", "model"], as_index=False)
        .agg(nmse_mean=("nmse", "mean"), nmse_std=("nmse", "std"))
    )
    noise_summary["nmse_std"] = noise_summary["nmse_std"].fillna(0.0)

    ridge_df = (
        noise_summary[noise_summary["model"] == "Ridge"]
        .rename(columns={"nmse_mean": "ridge_mean", "nmse_std": "ridge_std"})
        .drop(columns=["model"])
    )
    mlqrc_df = (
        noise_summary[noise_summary["model"] == "ML-QRC-M"]
        .rename(columns={"nmse_mean": "mlqrc_mean", "nmse_std": "mlqrc_std"})
        .drop(columns=["model"])
    )
    noise_table = ridge_df.merge(mlqrc_df, on="p_dep", how="inner").sort_values("p_dep")
    noise_table["gap_ridge_minus_mlqrc"] = noise_table["ridge_mean"] - noise_table["mlqrc_mean"]

    raw_results.to_csv(raw_cache_csv, index=False, float_format="%.6f")
    noise_summary.to_csv(summary_cache_csv, index=False, float_format="%.6f")
    noise_table.to_csv(wide_cache_csv, index=False, float_format="%.6f")
    raw_results.to_csv(raw_csv, index=False, float_format="%.6f")
    noise_summary.to_csv(summary_csv, index=False, float_format="%.6f")
    noise_table.to_csv(wide_csv, index=False, float_format="%.6f")

    print(f"Finished noise sweep in {(time.time() - t0) / 60.0:.1f} minutes")
    print(f"Saved raw results to: {raw_csv.resolve()}")
    print(f"Saved summary to: {summary_csv.resolve()}")
    print(f"Saved table to: {wide_csv.resolve()}")

display(noise_table)


In [ ]:
# This cell prints the draft-ready summary table as mean ± std.

print("=" * 78)
print(" Noise Robustness on Mackey-Glass  (5 seeds)")
print("=" * 78)
print(f"  {'p_dep':>6}   {'Ridge':>23}   {'ML-QRC-M (ours)':>23}   {'Gap':>10}")
print("-" * 78)
for _, row in noise_table.iterrows():
    print(
        f"  {row['p_dep']:6.3f}   "
        f"{row['ridge_mean']:.4f} ± {row['ridge_std']:.4f}   "
        f"{row['mlqrc_mean']:.4f} ± {row['mlqrc_std']:.4f}   "
        f"{row['gap_ridge_minus_mlqrc']:+.4f}"
    )
print("=" * 78)

print("\nLaTeX rows:")
for _, row in noise_table.iterrows():
    print(
        f"    {row['p_dep']:.3f} & "
        f"${row['ridge_mean']:.4f}\pm{row['ridge_std']:.4f}$ & "
        f"$\mathbf{{{row['mlqrc_mean']:.4f}\pm{row['mlqrc_std']:.4f}}}$ & "
        f"${row['gap_ridge_minus_mlqrc']:+.4f}$ \\"
    )


In [ ]:
# This cell draws the two-line noise-robustness figure requested by the draft.

ridge_curve = noise_table.sort_values("p_dep")
x = ridge_curve["p_dep"].to_numpy(dtype=np.float32)

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=1000)

ax.plot(
    x,
    ridge_curve["ridge_mean"].to_numpy(),
    marker="o",
    linewidth=1.0,
    markersize=3.0,
    color="#74c476",
    label="Ridge",
)
ax.fill_between(
    x,
    ridge_curve["ridge_mean"].to_numpy() - ridge_curve["ridge_std"].to_numpy(),
    ridge_curve["ridge_mean"].to_numpy() + ridge_curve["ridge_std"].to_numpy(),
    color="#74c476",
    alpha=0.15,
)

ax.plot(
    x,
    ridge_curve["mlqrc_mean"].to_numpy(),
    marker="o",
    linewidth=1.0,
    markersize=3.0,
    color="#005a32",
    label="ML-QRC-M",
)
ax.fill_between(
    x,
    ridge_curve["mlqrc_mean"].to_numpy() - ridge_curve["mlqrc_std"].to_numpy(),
    ridge_curve["mlqrc_mean"].to_numpy() + ridge_curve["mlqrc_std"].to_numpy(),
    color="#005a32",
    alpha=0.15,
)

ax.set_xlabel(r"$p_{dep}$")
ax.set_ylabel("NMSE")
ax.set_title("Noise Robustness")
ax.legend(fontsize=5.5, frameon=False)

fig_pdf = ARTIFACT_DIR / "fig_noise.pdf"
fig_png = ARTIFACT_DIR / "fig_noise.png"
plt.tight_layout()
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {fig_pdf.resolve()}")
print(f"Saved figure to: {fig_png.resolve()}")


In [ ]:
# This cell prints a short textual interpretation of the sweep so the
# notebook output is easy to reuse when updating the draft prose.

best_gap_idx = int(np.argmax(noise_table["gap_ridge_minus_mlqrc"].to_numpy()))
worst_gap_idx = int(np.argmin(noise_table["gap_ridge_minus_mlqrc"].to_numpy()))

best_row = noise_table.iloc[best_gap_idx]
worst_row = noise_table.iloc[worst_gap_idx]

print("Largest ML-QRC advantage:")
print(
    f"  p_dep={best_row['p_dep']:.3f}  "
    f"Ridge={best_row['ridge_mean']:.6f}  "
    f"ML-QRC-M={best_row['mlqrc_mean']:.6f}  "
    f"gap={best_row['gap_ridge_minus_mlqrc']:+.6f}"
)
print()
print("Weakest ML-QRC advantage:")
print(
    f"  p_dep={worst_row['p_dep']:.3f}  "
    f"Ridge={worst_row['ridge_mean']:.6f}  "
    f"ML-QRC-M={worst_row['mlqrc_mean']:.6f}  "
    f"gap={worst_row['gap_ridge_minus_mlqrc']:+.6f}"
)


## Output Files

The notebook writes:

- `noise_robustness_raw.csv`: one NMSE row per seed, model, and noise level
- `noise_robustness_summary.csv`: long-format mean/std summary
- `table_noise.csv`: wide-format draft table
- `fig_noise.pdf` and `fig_noise.png`: two-line noise-robustness plot

All files are saved under `notebooks/artifacts/noise_robustness/`.
